# EcoCharge Dublin — K-Means Clustering
**Author:** Anqi Xie  
**Branch:** ml  
**Description:** K-Means clustering to recommend optimal new EV charging station locations in Dublin, based on traffic volume, existing charger supply, road density, and renewable energy score.

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Load feature dataset
df = pd.read_csv('unified_features.csv')

print(f'Total records: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print('=== Basic Statistics ===')
print(df.describe())

# Check for missing values
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['traffic_volume'], bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Traffic Volume Distribution')
axes[0, 0].set_xlabel('Traffic Volume')

axes[0, 1].hist(df['charger_count_nearby'], bins=15, color='green', edgecolor='white')
axes[0, 1].set_title('Charger Count Nearby Distribution')
axes[0, 1].set_xlabel('Charger Count')

axes[1, 0].hist(df['road_density'], bins=30, color='orange', edgecolor='white')
axes[1, 0].set_title('Road Density Distribution')
axes[1, 0].set_xlabel('Road Density')

axes[1, 1].hist(df['renewable_score'], bins=20, color='purple', edgecolor='white')
axes[1, 1].set_title('Renewable Score Distribution')
axes[1, 1].set_xlabel('Renewable Score')

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Gap Score Calculation
Gap Score represents the supply-demand gap at each location:
- High traffic volume = high EV charging demand
- Low charger count = low supply
- High gap score = priority location for new charger

In [ ]:
# Calculate gap score: demand / (supply + 1)
df['gap_score'] = df['traffic_volume'] / (df['charger_count_nearby'] + 1)

# Normalised gap score (0-1 scale, easier to interpret)
df['gap_score_normalised'] = (df['gap_score'] - df['gap_score'].min()) / (df['gap_score'].max() - df['gap_score'].min())

print('Gap Score Statistics:')
print(df['gap_score'].describe())
print('\nNormalised Gap Score Statistics:')
print(df['gap_score_normalised'].describe())

# Visualise both distributions side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['gap_score'], bins=30, color='crimson', edgecolor='white')
axes[0].set_title('Gap Score Distribution')
axes[0].set_xlabel('Gap Score (Traffic / (Chargers + 1))')
axes[0].set_ylabel('Count')

axes[1].hist(df['gap_score_normalised'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Normalised Gap Score Distribution')
axes[1].set_xlabel('Normalised Gap Score (0-1)')
axes[1].set_ylabel('Count')

plt.suptitle('Gap Score Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gap_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Filter top 20% highest gap score locations
threshold = df['gap_score'].quantile(0.80)
df_priority = df[df['gap_score'] >= threshold].copy()

print(f'Total locations: {len(df)}')
print(f'Priority locations (top 20% gap score): {len(df_priority)}')
print(f'Gap score threshold: {threshold:.2f}')

## 4. Feature Scaling

In [ ]:
# Select features for clustering
feature_cols = [
    'traffic_volume',
    'charger_count_nearby',
    'road_density',
    'gap_score',
    'renewable_score'  
]
# Standardise features
scaler = StandardScaler()
scaled = scaler.fit_transform(df_priority[feature_cols])

print('Features scaled successfully.')
print(f'Shape: {scaled.shape}')

## 5. Elbow Method — Finding Optimal K

In [ ]:
inertia = []
k_range = range(2, 15)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    inertia.append(km.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method — Optimal K Selection')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. K-Means Clustering

In [ ]:
# Run K-Means with K=10 (minimum requirement: ≥10 recommended locations)
K = 10
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(scaled)

df_priority['cluster'] = kmeans.labels_

print(f'K-Means completed with K={K}')
print('\nCluster sizes:')
print(df_priority['cluster'].value_counts().sort_index())

## 7. Silhouette Score — Evaluating Cluster Quality

In [ ]:
score = silhouette_score(scaled, kmeans.labels_)
print(f'Silhouette Score: {score:.4f}')
print('(Score range: -1 to 1. Closer to 1 = better clustering quality)')

## 8. Recommended Locations — Cluster Centers

In [ ]:
# Get cluster center coordinates (use mean lat/lon of each cluster)
cluster_centers = df_priority.groupby('cluster').agg(
    lat=('lat', 'mean'),
    lon=('lon', 'mean'),
    avg_traffic=('traffic_volume', 'mean'),
    avg_gap_score=('gap_score', 'mean'),
    avg_chargers=('charger_count_nearby', 'mean'),
    site_count=('location_id', 'count')
).reset_index()

# Rank by average gap score (highest = most urgent)
cluster_centers = cluster_centers.sort_values('avg_gap_score', ascending=False).reset_index(drop=True)
cluster_centers['rank'] = cluster_centers.index + 1

print('=== Top 10 Recommended New EV Charger Locations ===')
print(cluster_centers[['rank', 'lat', 'lon', 'avg_traffic', 'avg_gap_score', 'avg_chargers']].to_string(index=False))

## 9. Map Visualisation

In [ ]:
# Colour palette for clusters
COLORS = [
    'red', 'blue', 'green', 'purple', 'orange',
    'darkred', 'cadetblue', 'darkgreen', 'darkpurple', 'darkblue'
]

# Create map
m = folium.Map(location=[53.33, -6.25], zoom_start=12)

# Plot all priority sites
for _, row in df_priority.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color=COLORS[int(row['cluster'])],
        fill=True,
        fill_opacity=0.5,
        popup=f"Site: {row['location_id']}<br>Traffic: {row['traffic_volume']:.0f}<br>Gap Score: {row['gap_score']:.1f}"
    ).add_to(m)

# Plot recommended locations (cluster centers)
for _, row in cluster_centers.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"Rank #{int(row['rank'])}<br>Avg Traffic: {row['avg_traffic']:.0f}<br>Avg Gap Score: {row['avg_gap_score']:.1f}",
        icon=folium.Icon(color='red', icon='bolt', prefix='fa')
    ).add_to(m)

m.save('kmeans_recommendations.html')
print('Map saved as kmeans_recommendations.html')
m

## 10. Summary

In [ ]:
print('=== K-Means Clustering Summary ===')
print(f'Total SCATS sites analysed:     {len(df)}')
print(f'Priority sites (top 20% gap):   {len(df_priority)}')
print(f'Number of clusters (K):         {K}')
print(f'Silhouette Score:               {score:.4f}')
print(f'Recommended new locations:      {len(cluster_centers)}')
print()
print('Top 3 priority locations:')
for _, row in cluster_centers.head(3).iterrows():
    print(f"  Rank #{int(row['rank'])}: ({row['lat']:.4f}, {row['lon']:.4f}) — Gap Score: {row['avg_gap_score']:.1f}")
